In [1]:
# setup cv
import os
os.environ["OPENCV_VIDEOIO_MSMF_ENABLE_HW_TRANSFORMS"] = "0"

# jupyter
from IPython.display import display, HTML

# datasets
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from huggingface_hub import HfApi

# policy
from lerobot.policies.act.modeling_act import ACTPolicy

# robots
from lerobot.robots.so101_follower import SO101FollowerConfig, SO101Follower

# record utils
from lerobot.utils.control_utils import init_keyboard_listener
from lerobot.utils.utils import log_say
from lerobot.utils.visualization_utils import _init_rerun
from lerobot.record import record_loop

# lerobot env

from lerobot.scripts.rl.gym_manipulator import ConvertToLeRobotObservation, BatchCompatibleWrapper

# torch
from torch import cuda

# utils
from dotenv import load_dotenv
import time
import pprint

# my code
from scripts.utils.paths import CALIBS_DIR, REPO_ROOT, DATASETS_DIR, HF_NAME, POLICIES_DIR, EVAL_DIR, OUTPUTS_DIR

# my env
from envs.so101_env import SO101Env

# set up os_env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# cuda
device = "cuda" if cuda.is_available() else "cpu"
print(f"Using device: {device}")

%load_ext autoreload
%autoreload 2

Using device: cuda


Set Params:

In [2]:
PUSH_TO_HUB     = False
SAVE_TO_DATASET = False
REPO_NAME         = 'so101_table_leg_move'
EXPERIMENT_NAME   = '50_episodes_v2'
POLICY_CHECKPOINT = '100000'
POLICY_TYPE = 'act'
NUM_EPISODES = 1
FPS = 30
EPISODE_TIME_SEC = 120
EVAL_TYPE = 'sim'

Log to hub if pushing:

In [3]:
if PUSH_TO_HUB:
    api = HfApi(token=os.getenv("HUGGINGFACE_TOKEN"))
    assert HF_NAME == api.whoami()['name']

Init Policy:

In [4]:
# Initialize the policy
policy_path = POLICIES_DIR / POLICY_TYPE / REPO_NAME / EXPERIMENT_NAME / 'checkpoints' / POLICY_CHECKPOINT / 'pretrained_model'
policy = ACTPolicy.from_pretrained(pretrained_name_or_path = policy_path, local_files_only=True)

Loading weights from local directory


In [5]:
display(HTML(f"""
<div style="max-height:300px; overflow:auto; border:1px solid #ccc; padding:10px;">
<pre>{pprint.pformat(policy.config, indent=2, width=80)}</pre>
</div>"""))

In [6]:
policy.unnormalize_outputs.buffer_action['mean']
# TODO fix normalization between action spaces

Parameter containing:
tensor([  4.1367, -27.3223,  49.3927,  15.2185, -57.1422,  15.8259],
       device='cuda:0')

Init env:

In [16]:
env = SO101Env(
    task = "TableLegAssembleTask",
    obs_type = "pixels_agent_pos",
)
# wrap to move to lerobot format
env = ConvertToLeRobotObservation(env, device = 'cuda')
env = BatchCompatibleWrapper(env)

In [17]:
obs,_ = env.reset()
obs['observation.images.top_cam']
obs['observation.state']

tensor([[0., 0., 0., 0., 0., 0.]], device='cuda:0')

Init dataset:

In [11]:
# configure the dataset features
if SAVE_TO_DATASET:
    action_features = hw_to_dataset_features(robot.action_features, "action")
    obs_features = hw_to_dataset_features(robot.observation_features, "observation")
    dataset_features = {**action_features, **obs_features}

    # create dataset
    policy_eval_path = EVAL_DIR / POLICY_TYPE / REPO_NAME / f"{EXPERIMENT_NAME}-{EVAL_TYPE}"

    kwargs = {
    "repo_id": f"{HF_NAME}/{REPO_NAME}_{EXPERIMENT_NAME}_{EVAL_TYPE}",
    "root"   : str(policy_eval_path)
    }

    if policy_eval_path.exists():
        dataset=LeRobotDataset(**kwargs)
    else:
        dataset = LeRobotDataset.create(
            **kwargs,
            fps=FPS,
            features=dataset_features,
            robot_type=robot.name,
            use_videos=True,
            image_writer_threads=4,
        )
        